In [1]:
import os
import numpy as np
import json
import pickle
import torch

import skimage
from skimage.transform import resize as cpu_resize
import imagesize
import imageio

import cv2

import matplotlib.pyplot as plt

from plyfile import PlyData,PlyElement

C0 = 0.28209479177387814
def RGB2SH(rgb):
    return (rgb - 0.5) / C0

def SH2RGB(sh):
    return sh * C0 + 0.5

def storePly(path, xyz, rgb):
    # Define the dtype for the structured array
    dtype = [('x', 'f4'), ('y', 'f4'), ('z', 'f4'),
            ('nx', 'f4'), ('ny', 'f4'), ('nz', 'f4'),
            ('red', 'u1'), ('green', 'u1'), ('blue', 'u1')]
    
    normals = np.zeros_like(xyz)

    elements = np.empty(xyz.shape[0], dtype=dtype)
    attributes = np.concatenate((xyz, normals, rgb), axis=1)
    elements[:] = list(map(tuple, attributes))

    # Create the PlyData object and write to file
    vertex_element = PlyElement.describe(elements, 'vertex')
    ply_data = PlyData([vertex_element])
    ply_data.write(path)

def write_ply(save_path,points,text=True):
    """
    save_path : path to save: '/yy/XX.ply'
    pt: point_cloud: size (N,3)
    """
    points = [(points[i,0], points[i,1], points[i,2]) for i in range(points.shape[0])]
    vertex = np.array(points, dtype=[('x', 'f4'), ('y', 'f4'),('z', 'f4')])
    el = PlyElement.describe(vertex, 'vertex', comments=['vertices'])
    PlyData([el], text=text).write(save_path)

def load_rgb(path: str, downscale = 1) -> np.ndarray:
    """ Load image

    Args:
        path (str): Given image file path
        downscale (numbers.Number, optional): Optional downscale ratio. Defaults to 1.

    Returns:
        np.ndarray: [H, W, 3], in range [0,1]
    """
    img = imageio.imread(path)
    img = skimage.img_as_float32(img)
    if downscale != 1:
        H, W, _ = img.shape
        img = cpu_resize(img, (int(H // downscale), int(W // downscale)), anti_aliasing=False)
    # [H, W, 3]
    return img

torch.set_printoptions(sci_mode=False)
np.set_printoptions(suppress=True)

def idx_to_frame_str(frame_index):
    return f'{frame_index:08d}'

def idx_to_img_filename(frame_index):
    return f'{idx_to_frame_str(frame_index)}.jpg'

def idx_to_lidar_filename(frame_index):
    return f'{idx_to_frame_str(frame_index)}.npz'

def idx_to_mask_filename(frame_index, compress=True):
    ext = 'npz' if compress else 'npy'
    return f'{idx_to_frame_str(frame_index)}.{ext}'

cityscapes_classes = [
    'road', 'sidewalk', 'building', 'wall', 'fence', 'pole',
    'traffic light', 'traffic sign', 'vegetation', 'terrain', 'sky',
    'person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle',
    'bicycle'
]
cityscapes_classes_ind_map = {cn: i for i, cn in enumerate(cityscapes_classes)}

cityscapes_dynamic_classes = [
    'person', 'rider', 'car', 'truck', 'bus', 'train', 'motorcycle', 'bicycle'
]

cityscapes_human_classes = [
    'person', 'rider'
]

base_path = "/SSD_DISK/users/yangzeyu/gaussian-splatting-4d/data/waymo/processed/segment-10061305430875486848_1080_000_1100_000_with_camera_labels"
scenario_fpath = os.path.join(base_path, "scenario.pt")
with open(scenario_fpath, 'rb') as f:
    scenario = pickle.load(f)

In [8]:
new_img_path = os.path.join(base_path, "images_undist")
new_mask_path = os.path.join(base_path, "masks_undist")
os.makedirs(new_img_path, exist_ok=True)
os.makedirs(new_mask_path, exist_ok=True)

c2ws = []
intrs = []
timestamps = []
hws = []

frames = []

cam_list = ['camera_FRONT', 'camera_FRONT_LEFT', 'camera_FRONT_RIGHT']
for cam_name in cam_list:
    n_frames = scenario['observers'][cam_name]['n_frames']
    cam_data = scenario['observers'][cam_name]['data']
    new_cam_path = os.path.join(new_img_path, cam_name)
    new_cam_mask_path = os.path.join(new_mask_path, cam_name)
    os.makedirs(new_cam_path, exist_ok=True)
    os.makedirs(new_cam_mask_path, exist_ok=True)
    for i in range(n_frames):
        c2w = cam_data['c2w'][i]
        intr = cam_data['intr'][i]
        distortion = cam_data['distortion'][i]
        timestamp = cam_data['timestamp'][i]
        hw = cam_data['hw'][i]
        im_path_frame = os.path.join(base_path, "images", cam_name, idx_to_img_filename(i))
        im_ori = load_rgb(im_path_frame) # H, W, 3 (0~1)
        h,  w = im_ori.shape[:2]
        newintr, roi = cv2.getOptimalNewCameraMatrix(intr, distortion, (w,h), 1, (w,h))
        img_undist = cv2.undistort(im_ori, intr, distortion, None, newintr)
        x, y, w, h = roi
        img_undist = img_undist[y:y+h, x:x+w]
        im_undist_path_frame = os.path.join(new_cam_path, idx_to_img_filename(i))
        cv2.imwrite(im_undist_path_frame, cv2.cvtColor((img_undist * 255).astype(np.uint8), cv2.COLOR_BGR2RGB))
        masks_path_frame = os.path.join(base_path, "masks", cam_name, idx_to_mask_filename(i))
        raw = np.load(masks_path_frame)['arr_0']
        ret = np.ones_like(raw).astype(np.bool8)
        ret[raw==cityscapes_classes_ind_map['sky']] = False
        mask_undist = cv2.undistort(ret.astype(img_undist.dtype), intr, distortion, None, newintr)
        mask_undist = mask_undist[y:y+h, x:x+w]
        mask_undist_path_frame = os.path.join(new_cam_mask_path, idx_to_img_filename(i))
        cv2.imwrite(mask_undist_path_frame, (mask_undist * 255).astype(np.uint8))
        c2w[:3, 1:3] *= -1
        frame = {
            'file_path': os.path.join("images_undist", cam_name, idx_to_img_filename(i)),
            'mask_path': os.path.join("masks_undist", cam_name, idx_to_img_filename(i)),
            'transform_matrix': c2w.tolist(),
            # 'time': cam_data['global_frame_ind'][i] / 10,
            'fl_x': newintr[0,0],
            'fl_y': newintr[1,1],
            'cx': newintr[0,2],
            'cy': newintr[1,2],
            'h': img_undist.shape[0],
            'w': img_undist.shape[1],
        }
        frames.append(frame)
        
        
out = {
    'frames': frames,
}
json_output_path = os.path.join(base_path, 'transforms_train.json')
with open(json_output_path, 'w') as f:
    json.dump(out, f, indent=2)

/home/yangzeyu/anaconda3/envs/open-mmlab/lib/python3.7/site-packages/ipykernel_launcher.py:62: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.


In [173]:
lidar_list = ['lidar_TOP']
points_all = []

for lidar_name in lidar_list:
    n_frames = scenario['observers'][lidar_name]['n_frames']
    lidar_data = scenario['observers'][lidar_name]['data']
    lidar_path = os.path.join(base_path, "lidars", lidar_name)
    for i in range(0, n_frames, 5):
        fpath =os.path.join(lidar_path, idx_to_lidar_filename(i))
        ret = np.load(fpath)
        lidar_gt = dict(
            rays_o = torch.tensor(ret['rays_o'].reshape(-1, 3), dtype=torch.float),
            rays_d = torch.tensor(ret['rays_d'].reshape(-1, 3), dtype=torch.float),
            ranges = torch.tensor(ret['ranges'].reshape(-1), dtype=torch.float),
        )
        pts_local = torch.addcmul(lidar_gt['rays_o'], lidar_gt['rays_d'], lidar_gt['ranges'].unsqueeze(-1))
        pts_local = pts_local.numpy()
        rotation = lidar_data['l2w'][i, :3, :3]
        translation = lidar_data['l2w'][i, :3, 3]
        pts_global = ((rotation @ pts_local.T) + translation[:,None]).T
        points_all.append(pts_global)
points_all = np.concatenate(points_all, 0)
num_pts = points_all.shape[0]
shs = np.random.random((num_pts, 3)) / 255.0

ply_path = os.path.join(base_path, "points3d.ply")
storePly(ply_path, points_all, SH2RGB(shs) * 255)

numpy.uint8